# Cryptocurrency Direction Prediction

**Project GIT Repository link**: https://github.com/Tan1208257/Crypto_Price_Detection.git

I have approached this project in 2 ways.
1. Build and evaluate machine learning models to predict the next 1-hour price direction (UP/DOWN) for individual cryptocurrencie datasets using engineered technical features for each dataset individually to analyse individual insights.
2. Build and evaluate machine learning models to predict the next 1-hour price direction (UP/DOWN) on a merged dataset for learning generalized cross-asset patterns.

**The goal is to determine the best performing model per asset as well as when merged and to analyze feature behavior.**

### I. Individual Dataset Approach

Each notebook uses one dataset of 15-minute OHLCV data for a specific crypto pair.
The datasets are:
1. Bitcoin (BTCUSDT)
2. Ethereum (ETHUSDT)
3. Solana (SOLUSDT)
4. Dogecoin (DOGEUSDT)
5. XRP (XRPUSDT)

### Data Preprocessing
### **1. Resampling**

Original frequency: 15-minute candles

Converted to: 1-hour candles

Aggregation logic used: (**This preserves the true candlestick shape.**)
- Open -> First
- High -> Max
- Low -> Min
- Close -> Last
- Volume -> Sum
- Turnover -> Sum
This preserves OHLC structure correctly. I tried averaged prices instead but then the candle structure would be incorrect and technical indicators would become invalid.

Resampling 15-min candles to 1-hour candles where done as cryptocurrency markets are highly noisy in low timeframes. When aggregated to 1-hour, clearer trend patterns emerge and indicators become more stable.
Also they are more economically meaningful and more relevant for algorithmic trading stratergies

#### **2. Target Variable Creation**

Binary classification target:
$$
y_t =
\begin{cases}
1 & \text{if } Close_{t+1} > Close_t \\
0 & \text{otherwise}
\end{cases}
$$

This is done so that the model predicts next hour direction.

### Feature Engineering
#### **1. Return-Based Features**
- % return of Close
- % return of Open
- % return of High/Low
- % change in Volume
- % change in Turnover

Raw price levels were avoided because crypto assets vary massively in scale (BTC vs DOGE). Returns make models asset-independent.

#### **2. Candle Structure Features**
- High-Low range %
- Open-Close body %

These features capture volatility compression and expansion phases. Because in case of BTC and ETH, range features showed stronger predictive power compared to DOGE.

#### **3. Technical Indicators**

Manually implemented:
- RSI (Relative Strength Index): Momentum strength indicator.
- Bollinger Band Z-Score: Distance from moving average normalized by volatility.
- ATR (Average True Range): Measures volatility expansion.


RSI and MACD features consistently appeared in top feature importance rankings. ATR was more relevant for SOL and DOGE (higher volatility assets).

### Train-Test Strategy

Time-series safe split:
- Last 1 year → Test set
- Earlier data → Train set

No random shuffle so that it prevents look-ahead bias.

Cross-validation method used: TimeSeriesSplit (5 folds). **This preserves chronological order**

### Models Used
I used various models for each datasets. The models mainly used were: Logistic Regression, Random Forest, HistGradientBoosting, XGBoost.

Scoring metric: ROC-AUC
- Best performing model across all 5 assets.
- Consistently highest ROC-AUC.

### Model Evaluation Metrics
Metrics used:
- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

Crypto direction is nearly balanced (≈50/50). ROC-AUC measures ranking quality rather than thresholded classification.

Across assets:

- Logistic Regression -> Weak baseline
- Random Forest	-> Moderate
- HistGradientBoosting	-> Strong
- XGBoost (Tuned) -> Best

### Important Analysis

- For BTC & ETH: 
There was strong influence from long-window momentum. More stable trend-following behavior.
- For SOL & DOGE
Higher importance of short-term lags. More reactive, volatility-driven structure.
- For XRP
Mixed behavior, less consistent pattern. Lower predictability compared to BTC.
- Large-cap coins show smoother, trend-driven behavior.

### II. Merged Dataset Approach

Instead of modeling asset-specific behavior, this approach attempts:
- Cross-asset learning
- Shared feature structure discovery
- Increased data volume for better generalization

### Data Preprocessing
### **1. Individual Preprocessing (Per Asset)**

Each dataset undergoes the same pipeline as in the separate notebooks:
- 15-minute → 1-hour resampling
- Proper OHLC aggregation
- Feature engineering
- Target creation (next-hour direction)
So the preprocessing logic remains consistent.

#### **2. Merging Process**

- All 5 datasets are concatenated vertically
- A new column (e.g., symbol) identifies the asset

#### **3. Target Variable**

Binary classification target:
$$
y_t =
\begin{cases}
1 & \text{if } Close_{t+1} > Close_t \\
0 & \text{otherwise}
\end{cases}
$$

The classification task remains identical.

### Train-Test Strategy

Time-series safe split is maintained.
- Latest time period → Test set
- Earlier period → Train set
- TimeSeriesSplit for cross-validation
- This prevents leakage across time.

### Models Used
I used various models for each datasets. The models mainly used were: Logistic Regression, Random Forest, HistGradientBoosting, XGBoost.

Scoring metric: ROC-AUC
- Best performing model across the merged dataset.
- Consistently highest ROC-AUC.

### Model Evaluation Metrics
Metrics used:
- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

**Compared to the individual datasets assessment**, the merged dataset models:
- Logistic Regression -> Slight improvement vs separate
- Random Forest -> More stable
- HistGradientBoosting -> Best performer
- XGBoost -> Strong

The merged HistGradientBoosting model performs:
- Equal or slightly better than separate models
- More stable across assets

## Key Differences Compared to Separate Modeling
1. Increased Dataset Size
- Merged dataset: 5× more observations, More robust learning, Reduced variance

**Larger sample size improves estimator stability.**

2. Cross-Asset Pattern Learning
- The model can learn: Common volatility responses, Universal momentum effects, Shared technical indicator behavior

For Example: If RSI overbought behavior works for BTC and ETH, the model can generalize it to SOL.

3. Regularization Effect
- More data naturally:
- Reduces overfitting
- Improves generalization

**Makes tree models more stable.**